In [18]:
#Сначала устанавливаем необходимые библиотеки jieba (для сегментации китайского текста), scikit-learn (TF-IDF и кластеризация)
!pip install jieba scikit-learn

#1. Импортируем библиотеки 
import re # Для удаления всего, кроме китайских иероглифов, чистим текст от мусора
import jieba # Для сегментации китайского текста 
import numpy as np # Для работы с числами и матрицами 
import pandas as pd # Для удобного вывода таблиц 
import requests 
from sklearn.feature_extraction.text import TfidfVectorizer # Превращает текст в числа (TF-IDF матрица)

# Загружаем стоп-слова из GitHub, можем и сами создать, если есть желание
url = "https://raw.githubusercontent.com/goto456/stopwords/master/cn_stopwords.txt"
response = requests.get(url)
stopwords = set(response.text.split())
print(f"Загружено стоп-слов: {len(stopwords)}")

# Открываем файлы
files = [
    "Текст_Красные Копья.txt",
    "Текст_Цинбан.txt"
]

documents = []

for filename in files:
    with open(filename, "r", encoding="utf-8") as f:
        text = f.read()
    
    # Сегментация
    words = jieba.lcut(text)
    
    # Очистка: оставляем только иероглифы, убираем стоп-слова
    cleaned = []
    for w in words:
        w_clean = re.sub(r'[^\u4e00-\u9fff]', '', w)
        if w_clean and w_clean not in stopwords:
            cleaned.append(w_clean)
    
    # Объединяем обратно в текст
    documents.append(" ".join(cleaned))
    print(f"{filename}: {len(cleaned)} слов")

print(f"\nВсего загружено документов: {len(documents)}")

Загружено стоп-слов: 746
Текст_Красные Копья.txt: 13530 слов
Текст_Цинбан.txt: 2872 слов

Всего загружено документов: 2


In [17]:
#TF-IDF, сравниваем содержание статей

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Превращаем тексты в числа
vectorizer = TfidfVectorizer(max_features=100)
tfidf_matrix = vectorizer.fit_transform(documents)
feature_names = vectorizer.get_feature_names_out()

# Получаем веса слов для каждой статьи
dense = tfidf_matrix.toarray()
       
print("\n20 КЛЮЧЕВЫХ СЛОВ ДЛЯ КАЖДОЙ СТАТЬИ\n")

# Названия для статей
labels = ["Красные копья (红枪会)", "Цинбан (青帮)"]

for i in range(len(documents)):
    weights = dense[i]
    # Находим 20 слов с самыми большими весами
    top_indices = weights.argsort()[-20:][::-1]
    
    print(f"{labels[i]}:")
    for idx in top_indices:
        print(f"   {feature_names[idx]}: {weights[idx]:.4f}")
    print()

# Основные слова

print("СЛОВА-МАРКЕРЫ\n")

# Считаем разницу для каждого слова
differences = []
for j, word in enumerate(feature_names):
    weight_1 = dense[0][j]  # Красные копья
    weight_2 = dense[1][j]  # Цинбан
    diff = abs(weight_1 - weight_2)
    differences.append((word, weight_1, weight_2, diff))

# Сортируем по размеру различия
differences.sort(key=lambda x: x[3], reverse=True)

print(f"{'Слово':<12} {'Красные копья':<18} {'Цинбан':<12} {'Разница':<8}")
print("-" * 55)
for word, w1, w2, diff in differences[:15]:
    print(f"{word:<12} {w1:<18.4f} {w2:<12.4f} {diff:<8.4f}")


#Итог

with open("tfidf_result.txt", "w", encoding="utf-8") as f:
    f.write("20 КЛЮЧЕВЫХ СЛОВ\n\n")
    
    for i in range(len(documents)):
        f.write(f"{labels[i]}:\n")
        weights = dense[i]
        top_indices = weights.argsort()[-20:][::-1]
        for idx in top_indices:
            f.write(f"   {feature_names[idx]}: {weights[idx]:.4f}\n")
        f.write("\n")
    
    f.write("\nСЛОВА-МАРКЕРЫ\n")
    f.write(f"{'Слово':<12} {'Красные копья':<18} {'Цинбан':<12}\n")
    f.write("-" * 45 + "\n")
    for word, w1, w2, diff in differences[:20]:
        f.write(f"{word:<12} {w1:<18.4f} {w2:<12.4f}\n")

print("\nРезультат сохранён в файл 'tfidf_result.txt'")


20 КЛЮЧЕВЫХ СЛОВ ДЛЯ КАЖДОЙ СТАТЬИ

Красные копья (红枪会):
   民团: 0.6072
   土匪: 0.3492
   民国: 0.3453
   地方: 0.2443
   组织: 0.1826
   武装: 0.1587
   自卫: 0.1429
   台湾: 0.1389
   山东: 0.1349
   影印: 0.1191
   铅本: 0.1072
   成文: 0.1072
   匪患: 0.0972
   剿匪: 0.0952
   政府: 0.0893
   军队: 0.0873
   近代: 0.0873
   团丁: 0.0853
   乡民: 0.0853
   出版社: 0.0847

Цинбан (青帮):
   读书: 0.6221
   张啸林: 0.5554
   林怀部: 0.3332
   白城市: 0.3332
   没有: 0.1581
   一个: 0.1423
   时间: 0.1265
   需要: 0.0790
   进行: 0.0474
   不能: 0.0474
   乡村: 0.0474
   当时: 0.0474
   一定: 0.0316
   地方: 0.0316
   成为: 0.0316
   社会: 0.0316
   方面: 0.0158
   政治: 0.0158
   出版社: 0.0158
   重要: 0.0158

СЛОВА-МАРКЕРЫ

Слово        Красные копья      Цинбан       Разница 
-------------------------------------------------------
读书           0.0000             0.6221       0.6221  
民团           0.6072             0.0000       0.6072  
张啸林          0.0000             0.5554       0.5554  
土匪           0.3492             0.0000       0.3492  
民国           0.3453  